KD_LAMBDA      = 0.5   
KD_TEMPERATURE = 4.0

In [8]:
# ============================================================
# S2: STUDENT + KD (no artifact gate)
#
#                 PSG
#                  |
#          Frozen Teacher (MultiScaleSleepNetPlain, 5-seed)
#                  |
#            soft probability
#                  |
#                  | KD
#                  v
#   Zmax --> Student --> logits
#   E4   --> Student
#
#   Loss = L_CE(student, true_label) + lambda_KD * L_KD(student, teacher)
#
# UPDATED: uses the 3-way split (_train_subs.npy / _val_subs.npy /
# _test_subs.npy). Checkpoint selection uses VAL F1 every epoch;
# TEST is evaluated exactly once, after training, using the
# val-selected best checkpoint. IMPORTANT: the teacher checkpoints
# (TEACHER_CKPT_DIR) must have been trained on subjects that don't
# overlap with this script's VAL/TEST sets, or the teacher's soft
# labels would leak val/test information into student training --
# make sure TEACHER_CKPT_DIR points to a teacher trained on the
# SAME SPLIT_PATH split.
#
# Architecture is IDENTICAL to S1 (WearableBaselineModel, simple
# concat fusion, no gate) so any improvement over S1 can be
# attributed purely to the KD signal, not to an architecture change.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
PSG_EEG_PATH      = r"D:\22\AA\preprocess\preprocessed_FFinal"             # teacher input (.npz)
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"    # student input (.npz)
SPLIT_PATH        = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"  # _train/_val/_test_subs.npy
TEACHER_CKPT_DIR  = r"D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed"    # frozen teacher checkpoints
EVAL_PATH         = r"D:\22\AA\AA journal\evaluation\teacher-student\s2_kd_student"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7
WINDOW      = 2 * CONTEXT + 1
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS         = [42, 123, 256, 789, 999]
TEACHER_SEEDS = [42, 123, 256, 789, 999]

# --- Teacher config (must match MultiScaleSleepNetPlain exactly) ---
T_D_MODEL = 128
T_DROPOUT = 0.4
T_N_HEADS = 4

# --- Student config (identical to S1 -- same architecture) ---
S_D_MODEL = 96
S_DROPOUT = 0.3

KD_LAMBDA      = 0.5   # weight on the KD term (added on top of CE, not blended)
KD_TEMPERATURE = 4.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"S2: Student + KD (no artifact gate)")
print(f"KD lambda     : {KD_LAMBDA}   Temperature: {KD_TEMPERATURE}")
print(f"Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)")
print(f"Student       : same architecture as S1 (WearableBaselineModel)")
print(f"Output        : {EVAL_PATH}")


# ============================================================
# ============  TEACHER ARCHITECTURE (frozen)  ================
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=T_D_MODEL, dropout=T_DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=T_D_MODEL, n_heads=T_N_HEADS, dropout=T_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


class MultiScaleSleepNetPlain(nn.Module):
    """Teacher architecture -- identical to your trained checkpoints."""
    def __init__(
        self, in_ch=3, d_model=T_D_MODEL, n_layers=2,
        dropout=T_DROPOUT, n_classes=5, context=CONTEXT
    ):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T)).permute(0, 2, 1)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)
        inter = self.inter_blocks(epoch_feat + self.inter_pos)
        center = inter[:, self.context, :]
        return self.classifier(center)


# ============================================================
# ============  STUDENT ARCHITECTURE (identical to S1)  =======
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    def __init__(self, in_ch=2, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class E4Encoder(nn.Module):
    def __init__(self, in_ch=3, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class WearableBaselineModel(nn.Module):
    """IDENTICAL to S1 -- simple concat fusion, no gate."""
    def __init__(self, d_model=S_D_MODEL, n_classes=5, dropout=S_DROPOUT):
        super().__init__()
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        z = self.zmax_enc(zmax_x)
        e = self.e4_enc(e4_x)
        fused = torch.cat([z, e], dim=1)
        logits = self.classifier(fused)
        return logits


# ============================================================
# COMBINED DATASET: teacher input (PSG, context window) +
# student input (Zmax + E4, single center epoch)
# ============================================================
class KDDataset(Dataset):
    def __init__(self, subject_list, psg_path, student_path, context=CONTEXT):
        self.context = context
        self.data = []
        self.index = []
        skipped_mismatch = 0

        for sub in subject_list:
            psg_fp = os.path.join(psg_path, f"{sub}.npz")
            stu_fp = os.path.join(student_path, f"{sub}.npz")
            if not (os.path.exists(psg_fp) and os.path.exists(stu_fp)):
                continue

            with np.load(psg_fp) as d:
                eeg = d['eeg'][:, [0, 1], :]
                eog = d['eog'][:, [0], :]
                psg_signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                psg_labels = d['labels'].copy()

            with np.load(stu_fp) as d:
                zmax_arr = d['zmax_eeg']
                e4_arr   = d['e4']
                stu_labels = d['labels'].copy()

            n = min(len(psg_labels), len(stu_labels))
            if abs(len(psg_labels) - len(stu_labels)) > 5:
                skipped_mismatch += 1

            sub_idx = len(self.data)
            self.data.append((psg_signal, psg_labels, zmax_arr, e4_arr, stu_labels))
            for i in range(n):
                self.index.append((sub_idx, i, n))

        print(f"  Subjects loaded: {len(self.data)}  (mismatches>5 epochs: {skipped_mismatch})")
        print(f"  Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        psg_signal, psg_labels, zmax_arr, e4_arr, stu_labels = self.data[sub_idx]

        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(psg_signal[ei])
        psg_x = np.stack(window_epochs, axis=0)

        ei = min(center_i, zmax_arr.shape[0] - 1, e4_arr.shape[0] - 1)
        zmax_x = zmax_arr[ei]
        e4_x   = e4_arr[ei]
        y = int(stu_labels[center_i]) if center_i < len(stu_labels) else int(psg_labels[center_i])

        return (
            torch.FloatTensor(psg_x),
            torch.FloatTensor(zmax_x),
            torch.FloatTensor(e4_x),
            torch.tensor(y, dtype=torch.long),
        )


# ============================================================
# 3-WAY SPLIT + BUILD DATASETS
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not found at {SPLIT_PATH}")

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")

print("\nBuilding KD datasets...")
print("Train:")
train_ds = KDDataset(TRAIN_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Val:")
val_ds   = KDDataset(VAL_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Test:")
test_ds  = KDDataset(TEST_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(f"{name}_ds has 0 samples! Check PSG_EEG_PATH / STUDENT_DATA_PATH.")
print("Datasets ready.")


# ============================================================
# LOAD FROZEN TEACHER ENSEMBLE
# ============================================================
print(f"\nLoading {len(TEACHER_SEEDS)} frozen teacher checkpoints from {TEACHER_CKPT_DIR}...")
teachers = []
for seed in TEACHER_SEEDS:
    ckpt = os.path.join(TEACHER_CKPT_DIR, f"best_seed{seed}.pt")
    if not os.path.exists(ckpt):
        print(f"  MISSING: {ckpt}")
        continue
    m = MultiScaleSleepNetPlain(in_ch=3).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    teachers.append(m)
print(f"Loaded {len(teachers)} teacher models (frozen).")
if len(teachers) == 0:
    raise RuntimeError(f"No teacher checkpoints found in {TEACHER_CKPT_DIR}.")


@torch.no_grad()
def teacher_soft_targets(psg_x, temperature=KD_TEMPERATURE):
    probs_sum = None
    for m in teachers:
        logits = m(psg_x)
        probs = F.softmax(logits / temperature, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(teachers)


# ============================================================
# CLASS WEIGHTS -- from TRAIN only
# ============================================================
all_train_labels = []
for _, _, _, _, labs in train_ds.data:
    all_train_labels.extend(labs.tolist())
label_counts = np.array([Counter(all_train_labels).get(i, 1) for i in range(5)], dtype=np.float32)
cw = torch.FloatTensor(label_counts.sum() / (5 * label_counts)).to(device)
print(f"\nClass weights (from TRAIN set only): {dict(zip(LABEL_NAMES, cw.cpu().numpy().round(3)))}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def kd_loss_fn(student_logits, teacher_probs, labels, temperature, lam, class_weights):
    """L = L_CE + lambda * L_KD  (additive, NOT a convex blend)"""
    ce = F.cross_entropy(student_logits, labels, weight=class_weights)

    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    kd = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (temperature ** 2)

    total = ce + lam * kd
    return total, ce.item(), kd.item()


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total_ce = total_kd = 0
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, y in loader:
        psg_x, zmax_x, e4_x, y = psg_x.to(device), zmax_x.to(device), e4_x.to(device), y.to(device)
        optimizer.zero_grad()
        teacher_probs = teacher_soft_targets(psg_x)
        student_logits = model(zmax_x, e4_x)
        loss, ce_val, kd_val = kd_loss_fn(
            student_logits, teacher_probs, y, KD_TEMPERATURE, KD_LAMBDA, cw
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item(); total_ce += ce_val; total_kd += kd_val
        preds.extend(student_logits.argmax(1).cpu().numpy())
        labs_all.extend(y.cpu().numpy())

    n = len(loader)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    return total_loss / n, total_ce / n, total_kd / n, acc, f1


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, y in loader:
        logits = model(zmax_x.to(device), e4_x.to(device))
        preds.extend(logits.argmax(1).cpu().numpy())
        labs_all.extend(y.numpy())
    preds, labs_all = np.array(preds), np.array(labs_all)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs_all, preds)
    per_cls = f1_score(labs_all, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch", "train_loss", "train_ce", "train_kd", "train_acc", "train_f1_macro",
    "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "best_epoch", "best_val_f1", "acc", "f1_macro", "kappa",
          "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

all_results = []
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSTUDENT SEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=torch.Generator().manual_seed(seed))
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = WearableBaselineModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Student parameters: {n_params:,}  (teacher frozen, not counted)")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4, betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch  = -1
    best_path = os.path.join(EVAL_PATH, f"best_student_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_ce, tr_kd, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler)
        vl_acc, vl_f1, vl_kap, vl_per = evaluate(model, val_loader)

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch  = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f}(CE:{tr_ce:.3f}+KD:{tr_kd:.3f}) "
              f"TrF1:{tr_f1:.3f} | ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} Valk:{vl_kap:.3f}{tag}")

        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6), "train_ce": round(tr_ce, 6), "train_kd": round(tr_kd, 6),
                "train_acc": round(tr_acc, 6), "train_f1_macro": round(tr_f1, 6),
                "val_acc": round(vl_acc, 6), "val_f1_macro": round(vl_f1, 6), "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6), "val_f1_N1": round(vl_per[1], 6),
                "val_f1_N2": round(vl_per[2], 6), "val_f1_N3": round(vl_per[3], 6),
                "val_f1_REM": round(vl_per[4], 6),
                "lr": optimizer.param_groups[0]['lr'], "is_best": is_best,
            })

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate(model, test_loader)

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap, 'per_cls': fin_per,
                         'best_epoch': best_epoch, 'best_val_f1': best_val_f1})
    with open(csv_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed, "best_epoch": best_epoch, "best_val_f1": round(best_val_f1, 4),
            "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4), "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])

print(f"\n{'='*60}\nS2: STUDENT + KD (VAL-SELECTED, HONEST TEST) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")

print("\nCompare against your re-run S1 (on this same split) to measure")
print("the benefit of KD alone. Old numbers (57.09%/61.86% etc.) came")
print("from the leaked train/test-only split and are not comparable.")

print(f"\nEpoch log : {epoch_csv_path}")
print(f"Summary   : {csv_path}")
print("Done!")

Device        : cuda
S2: Student + KD (no artifact gate)
KD lambda     : 0.5   Temperature: 4.0
Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)
Student       : same architecture as S1 (WearableBaselineModel)
Output        : D:\22\AA\AA journal\evaluation\teacher-student\s2_kd_student
Split loaded -> Train:65  Val:11  Test:20

Building KD datasets...
Train:
  Subjects loaded: 61  (mismatches>5 epochs: 0)
  Samples: 57,021
Val:
  Subjects loaded: 10  (mismatches>5 epochs: 0)
  Samples: 9,742
Test:
  Subjects loaded: 19  (mismatches>5 epochs: 0)
  Samples: 18,898
Datasets ready.

Loading 5 frozen teacher checkpoints from D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed...
Loaded 5 teacher models (frozen).

Class weights (from TRAIN set only): {'Wake': np.float32(2.094), 'N1': np.float32(3.192), 'N2': np.float32(0.436), 'N3': np.float32(1.0), 'REM': np.float32(1.092)}

STUDENT SEED 42  (1/5)
  Student parameters: 153,989  (teacher frozen, not counted)
  Ep[01/3

for test


In [9]:
# ============================================================
# STEP A: CROSS-ATTENTION + MAMBA-STYLE STUDENT (NO TEACHER, NO KD)
#
#   Zmax EEG --> CNN --> SimplifiedMambaBlock --> z_seq (L, d)  --\
#                                                                  >-- CrossAttention(query=z, kv=e) --> Classifier
#   E4       --> CNN --> SimplifiedMambaBlock --> e_seq (L, d)  --/
#
#   Loss = plain weighted CE (NO KD, NO teacher) -- this isolates
#   the effect of the NEW ARCHITECTURE alone, so it can be
#   compared directly against S1 (WearableBaselineModel, concat
#   fusion, no Mamba, no attention) on the SAME val-fixed split.
#
# Direction of attention: Zmax is the QUERY, E4 is KEY/VALUE --
# i.e. EEG gets primacy and pulls relevant E4 information toward
# it (compensating when EEG itself is noisy).
#
# NOTE ON MAMBA IMPLEMENTATION: this uses a lightweight, pure
# PyTorch simplified state-space block (no mamba-ssm pip
# dependency), so it runs in any environment without CUDA/version
# issues. If you have your exact BiT-MamSleep Mamba block code,
# swap it in for `SimplifiedMambaBlock` below -- the rest of the
# script (data loading, cross-attention, training loop) is
# independent of which Mamba variant you use.
#
# Compare against your re-run S1 (on this same split) to see
# whether architecture alone (before any KD) already improves
# over the simple concat-fusion baseline.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
SPLIT_PATH        = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
EVAL_PATH         = r"D:\22\ii\evaluation\teacher-student\stepA_crossattn_mamba_noKD"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 96
DROPOUT = 0.3
D_STATE = 16     # SSM state dimension for the simplified Mamba block

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"STEP A: Cross-Attention + Mamba-style student (NO teacher, NO KD)")
print(f"Attention direction: Zmax (query) attends over E4 (key/value)")
print(f"Output : {EVAL_PATH}")


# ============================================================
# 3-WAY SPLIT
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not found at {SPLIT_PATH}")

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET (identical to S1 -- Zmax + E4, single center epoch)
# ============================================================
class WearableDataset(Dataset):
    def __init__(self, subject_list, data_path):
        self.data  = []
        self.index = []
        label_counter = Counter()

        n_loaded = 0
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                zmax_arr = d['zmax_eeg']    # (N, 2, 1920)
                e4_arr   = d['e4']          # (N, 3, 1920)
                labels   = d['labels'].copy()

            n = min(zmax_arr.shape[0], e4_arr.shape[0], len(labels))
            sub_idx = len(self.data)
            self.data.append((zmax_arr[:n], e4_arr[:n], labels[:n]))
            for i in range(n):
                self.index.append((sub_idx, i))
                label_counter[int(labels[i])] += 1
            n_loaded += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        print(f"  Subjects: {n_loaded}   Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, i = self.index[idx]
        zmax_arr, e4_arr, labels = self.data[sub_idx]
        zmax_x = zmax_arr[i]
        e4_x   = e4_arr[i]
        y      = int(labels[i])
        return torch.FloatTensor(zmax_x), torch.FloatTensor(e4_x), torch.tensor(y, dtype=torch.long)


print("\nBuilding datasets...")
print("Train:")
train_ds = WearableDataset(TRAIN_SUBS, STUDENT_DATA_PATH)
print("Val:")
val_ds   = WearableDataset(VAL_SUBS, STUDENT_DATA_PATH)
print("Test:")
test_ds  = WearableDataset(TEST_SUBS, STUDENT_DATA_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(f"{name}_ds has 0 samples! Check STUDENT_DATA_PATH.")
print("Datasets ready.")


# ============================================================
# SIMPLIFIED MAMBA-STYLE BLOCK (pure PyTorch, no mamba-ssm dep)
#
# This is a lightweight selective-SSM-inspired block: a gated
# linear recurrence with input-dependent gating, applied along
# the time axis. It captures long-range dependencies similarly
# in spirit to Mamba's selective state-space mechanism, without
# requiring the compiled mamba-ssm CUDA kernels.
#
# If you have your exact BiT-MamSleep Mamba block, replace this
# class -- everything downstream just expects
# SimplifiedMambaBlock(d_model) with forward(x: (B, L, d_model))
# -> (B, L, d_model).
# ============================================================
class SimplifiedMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, d_model * 2)   # splits into (x, gate)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)

        # Input-dependent selective parameters (the "selective" part of S6/Mamba)
        self.x_proj = nn.Linear(d_model, d_state * 2)     # -> (B_param, C_param)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)).unsqueeze(0))

        self.out_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, L, d_model)
        B, L, D = x.shape
        residual = x

        xz = self.in_proj(x)                 # (B, L, 2D)
        x_in, gate = xz.chunk(2, dim=-1)      # each (B, L, D)

        x_conv = self.conv(x_in.transpose(1, 2)).transpose(1, 2)  # (B, L, D)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))            # (B, L, D) -- selective timestep
        bc = self.x_proj(x_conv)                          # (B, L, 2*d_state)
        b_param, c_param = bc.chunk(2, dim=-1)             # each (B, L, d_state)

        A = -torch.exp(self.A_log)                        # (1, d_state), negative for stability

        # Sequential selective scan (simple, readable; fine for L~15-30 epoch windows)
        state = torch.zeros(B, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t, :].unsqueeze(-1)              # (B, D, 1)
            dA = torch.exp(dt_t * A.unsqueeze(1))         # (B, D, d_state)
            dB = dt_t * b_param[:, t, :].unsqueeze(1)      # (B, D, d_state)
            state = state * dA + dB * x_conv[:, t, :].unsqueeze(-1)
            y_t = (state * c_param[:, t, :].unsqueeze(1)).sum(-1)  # (B, D)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)                         # (B, L, D)

        y = y * F.silu(gate)
        out = self.out_proj(y)
        out = self.dropout(out)
        return self.norm(residual + out)


# ============================================================
# PER-EPOCH CNN + MAMBA ENCODER (single center epoch, no window
# in this Step-A version -- matches S1's single-epoch input;
# context-window can be added later like S5 did for the concat model)
# ============================================================
class ZmaxMambaEncoder(nn.Module):
    """CNN front-end (same design as S1's ZmaxEEGEncoder) + Mamba block."""
    def __init__(self, in_ch=2, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.mamba = SimplifiedMambaBlock(d_model, dropout=dropout)

    def forward(self, x):
        # x: (B, 2, 1920)
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)   # (B, L, d_model)
        return self.mamba(feat)   # (B, L, d_model) -- sequence preserved for attention


class E4MambaEncoder(nn.Module):
    """CNN front-end (same design as S1's E4Encoder) + Mamba block."""
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.mamba = SimplifiedMambaBlock(d_model, dropout=dropout)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)   # (B, L, d_model)
        return self.mamba(feat)


# ============================================================
# CROSS-ATTENTION FUSION
# Zmax = QUERY (EEG gets primacy), E4 = KEY/VALUE
# ============================================================
class CrossAttentionFusion(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=4, dropout=DROPOUT):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, zmax_seq, e4_seq):
        # zmax_seq: (B, L, d) is the QUERY -- EEG asks "what in E4 is relevant to me"
        # e4_seq:   (B, L, d) provides KEY and VALUE
        attn_out, attn_weights = self.attn(query=zmax_seq, key=e4_seq, value=e4_seq)
        fused = self.norm(zmax_seq + attn_out)
        fused = self.norm2(fused + self.ffn(fused))
        return fused, attn_weights


# ============================================================
# STEP-A MODEL: Cross-Attention + Mamba, NO teacher/KD
# ============================================================
class CrossAttnMambaStudent(nn.Module):
    def __init__(self, d_model=D_MODEL, n_classes=5, dropout=DROPOUT):
        super().__init__()
        self.zmax_enc = ZmaxMambaEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4MambaEncoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.fusion   = CrossAttentionFusion(d_model=d_model, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        z_seq = self.zmax_enc(zmax_x)   # (B, L, d)
        e_seq = self.e4_enc(e4_x)       # (B, L, d)
        fused_seq, _ = self.fusion(z_seq, e_seq)   # (B, L, d)
        pooled = fused_seq.mean(dim=1)              # (B, d) -- average pool over the sequence
        logits = self.classifier(pooled)
        return logits


# ============================================================
# CLASS WEIGHTS -- from TRAIN only
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights (from TRAIN set only):")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    preds, labs = [], []
    for zmax_x, e4_x, y in loader:
        zmax_x, e4_x, y = zmax_x.to(device), e4_x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(zmax_x, e4_x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1


@torch.no_grad()
def evaluate_fn(model, loader):
    model.eval()
    preds, labs = [], []
    for zmax_x, e4_x, y in loader:
        logits = model(zmax_x.to(device), e4_x.to(device))
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.numpy())
    preds = np.array(preds); labs = np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch", "train_loss", "train_acc", "train_f1_macro",
    "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
summary_fields = [
    "seed", "best_epoch", "best_val_f1",
    "test_acc", "test_f1_macro", "test_kappa",
    "test_f1_Wake", "test_f1_N1", "test_f1_N2", "test_f1_N3", "test_f1_REM",
]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, summary_fields).writeheader()


# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = CrossAttnMambaStudent().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params:,}")

    criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch  = -1
    best_path = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler, criterion
        )
        vl_acc, vl_f1, vl_kap, vl_per = evaluate_fn(model, val_loader)

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch  = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        lr = optimizer.param_groups[0]['lr']
        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f} "
              f"TrAcc:{tr_acc:.3f} TrF1:{tr_f1:.3f} | "
              f"ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} Valk:{vl_kap:.3f} "
              f"LR:{lr:.2e}{tag}")

        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6),
                "train_acc": round(tr_acc, 6),
                "train_f1_macro": round(tr_f1, 6),
                "val_acc": round(vl_acc, 6),
                "val_f1_macro": round(vl_f1, 6),
                "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6),
                "val_f1_N1":   round(vl_per[1], 6),
                "val_f1_N2":   round(vl_per[2], 6),
                "val_f1_N3":   round(vl_per[3], 6),
                "val_f1_REM":  round(vl_per[4], 6),
                "lr": lr,
                "is_best": is_best,
            })

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate_fn(model, test_loader)

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): "
          f"Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap,
        'per_cls': fin_per, 'best_epoch': best_epoch, 'best_val_f1': best_val_f1
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, summary_fields).writerow({
            "seed": seed, "best_epoch": best_epoch, "best_val_f1": round(best_val_f1, 4),
            "test_acc": round(fin_acc, 4), "test_f1_macro": round(fin_f1, 4),
            "test_kappa": round(fin_kap, 4),
            "test_f1_Wake": round(fin_per[0], 4), "test_f1_N1": round(fin_per[1], 4),
            "test_f1_N2": round(fin_per[2], 4), "test_f1_N3": round(fin_per[3], 4),
            "test_f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs   = np.array([r['acc'] for r in all_results]) * 100
f1s    = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s    = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nSTEP A: CROSS-ATTENTION + MAMBA (NO TEACHER/KD) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print("\nCompare against your re-run S1 (concat fusion, no Mamba, no")
print("attention, same split) to see if architecture alone helps")
print("BEFORE adding KD. If this beats S1, the next step is adding")
print("the frozen teacher + KD loss on top of THIS architecture.")

print(f"\nEpoch log : {epoch_csv_path}")
print(f"Summary   : {csv_summary_path}")
print("Done!")

Device : cuda
STEP A: Cross-Attention + Mamba-style student (NO teacher, NO KD)
Attention direction: Zmax (query) attends over E4 (key/value)
Output : D:\22\ii\evaluation\teacher-student\stepA_crossattn_mamba_noKD
Split loaded -> Train:65  Val:11  Test:20

Building datasets...
Train:
  Subjects: 61   Samples: 57,021
Val:
  Subjects: 10   Samples: 9,745
Test:
  Subjects: 19   Samples: 18,899
Datasets ready.

Class weights (from TRAIN set only):
  Wake: 2.094
  N1: 3.192
  N2: 0.436
  N3: 1.000
  REM: 1.092

SEED 42  (1/5)
  Parameters : 219,845
  Ep[01/30] Loss:1.350 TrAcc:0.439 TrF1:0.394 | ValAcc:0.458 ValF1:0.432 Valk:0.318 LR:3.33e-04 <- BEST
  Ep[02/30] Loss:1.156 TrAcc:0.529 TrF1:0.486 | ValAcc:0.496 ValF1:0.469 Valk:0.358 LR:5.00e-04 <- BEST
  Ep[03/30] Loss:1.070 TrAcc:0.573 TrF1:0.530 | ValAcc:0.547 ValF1:0.505 Valk:0.423 LR:4.97e-04 <- BEST
  Ep[04/30] Loss:1.010 TrAcc:0.601 TrF1:0.558 | ValAcc:0.574 ValF1:0.536 Valk:0.455 LR:4.91e-04 <- BEST
  Ep[05/30] Loss:0.976 TrAcc:0.619

KeyboardInterrupt: 